# INSTRUCTION-LEVEL PARALLELISM SIMULATOR (TAHAP 1: BASE + RAW HAZARD)
**Kontributor / Komit:** Anggota 1 (Kelompok 4)
**Mata Kuliah:** Organisasi dan Arsitektur Komputer
**Repositori GitHub:** https://github.com/aarrrryyaaaaa/K4_Organisasi-Arsitektur-Komputer.git

---
### Deskripsi Tahap 1
Notebook ini merupakan implementasi tahap pertama dari simulator ILP:
1. Mempertahankan struktur dasar simulator dari file acuan `K4_ILP.ipynb`.
2. Mengimplementasikan fungsi deteksi **RAW** (*Read After Write* / *True Data Dependency*).
3. Melakukan simulasi eksekusi sekuensial vs eksekusi paralel ILP yang mematuhi dependensi RAW.
4. Menampilkan visualisasi **Dependency Graph RAW** berstruktur tingkat eksekusi (Topological DAG).


In [ ]:
from dataclasses import dataclass
import matplotlib.pyplot as plt
import matplotlib.patches as patches

@dataclass
class Instruction:
    name: str
    destination: str
    sources: list

# Set Instruksi Dasar dengan Ketergantungan RAW
instructions = [
    Instruction('I1', 'R1', ['R2', 'R3']),
    Instruction('I2', 'R4', ['R5', 'R6']),
    Instruction('I3', 'R7', ['R1', 'R4']),     # RAW thd I1 (R1) dan I2 (R4)
    Instruction('I4', 'R8', ['R9', 'R10']),
    Instruction('I5', 'R11', ['R7', 'R8']),    # RAW thd I3 (R7) dan I4 (R8)
    Instruction('I6', 'R12', ['R13', 'R14']),  # Independen
]

# -------------------------------------------------
# Deteksi Ketergantungan RAW
# -------------------------------------------------
def has_raw_dependency(inst1, inst2):
    """
    Mengembalikan True jika inst2 membaca register yang ditulis oleh inst1 (RAW / True Dependency).
    """
    return inst1.destination in inst2.sources

def show_dependencies(instructions):
    print('\n' + '=' * 50)
    print('DEPENDENCY ANALYSIS (RAW - READ AFTER WRITE)')
    print('=' * 50)
    found = False
    for i in range(len(instructions)):
        for j in range(i + 1, len(instructions)):
            if has_raw_dependency(instructions[i], instructions[j]):
                print(f'{instructions[j].name} depends on {instructions[i].name} (RAW Register: {instructions[i].destination})')
                found = True
    if not found:
        print('No RAW dependencies found.')

# -------------------------------------------------
# Eksekusi Sekuensial
# -------------------------------------------------
def sequential_execution(instructions):
    print('\n' + '=' * 50)
    print('SEQUENTIAL EXECUTION')
    print('=' * 50)
    cycle = 0
    for inst in instructions:
        cycle += 1
        print(f'Cycle {cycle}: Executing {inst.name} ({inst.destination} = {" + ".join(inst.sources)})')
    print(f'\nTotal Sequential Cycles: {cycle}')
    return cycle

# -------------------------------------------------
# Eksekusi Paralel ILP (Mematuhi RAW)
# -------------------------------------------------
def parallel_execution(instructions):
    print('\n' + '=' * 50)
    print('INSTRUCTION-LEVEL PARALLEL EXECUTION (RAW RESTRICTIONS)')
    print('=' * 50)
    completed = set()
    remaining = instructions.copy()
    cycle = 0

    while remaining:
        cycle += 1
        current_cycle = []

        for instruction in remaining:
            satisfied = True
            for prev in instructions:
                if prev.name == instruction.name:
                    break
                if has_raw_dependency(prev, instruction):
                    if prev.name not in completed:
                        satisfied = False
                        break
            if satisfied:
                current_cycle.append(instruction)

        print(f'\nCycle {cycle}:')
        for inst in current_cycle:
            print(f'  -> {inst.name}: {inst.destination} = {" + ".join(inst.sources)}')

        for inst in current_cycle:
            completed.add(inst.name)
            remaining.remove(inst)

    print(f'\nTotal ILP Cycles: {cycle}')
    return cycle

# -------------------------------------------------
# Visualisasi Dependency Graph
# -------------------------------------------------
def draw_dependency_graph():
    fig, ax = plt.subplots(figsize=(10, 6.5), dpi=150)
    fig.patch.set_facecolor('#ffffff')
    ax.set_facecolor('#ffffff')
    levels = {
        'Level 1': (3.8, 'Eksekusi Cycle 1 - Operan Siap'),
        'Level 2': (2.0, 'Eksekusi Cycle 2 - Tunggu R1, R4'),
        'Level 3': (0.2, 'Eksekusi Cycle 3 - Tunggu R7, R8')
    }
    for lvl, (y, desc) in levels.items():
        ax.axhline(y=y, color='#e2e8f0', linestyle='--', linewidth=1)
        ax.text(-0.2, y, f'{lvl}\n({desc})', fontsize=8, fontweight='bold', color='#64748b', ha='right', va='center')
    nodes = {
        'I1': {'pos': (0.8, 3.8), 'code': 'R1 = R2 + R3'},
        'I2': {'pos': (2.8, 3.8), 'code': 'R4 = R5 + R6'},
        'I4': {'pos': (4.8, 3.8), 'code': 'R8 = R9 + R10'},
        'I6': {'pos': (6.6, 3.8), 'code': 'R12 = R13 + R14'},
        'I3': {'pos': (1.8, 2.0), 'code': 'R7 = R1 + R4'},
        'I5': {'pos': (3.8, 0.2), 'code': 'R11 = R7 + R8'},
    }
    edges = [
        ('I1', 'I3', 'RAW (R1)'),
        ('I2', 'I3', 'RAW (R4)'),
        ('I3', 'I5', 'RAW (R7)'),
        ('I4', 'I5', 'RAW (R8)'),
    ]
    for name, data in nodes.items():
        x, y = data['pos']
        rect = patches.FancyBboxPatch((x - 0.85, y - 0.35), 1.7, 0.7, boxstyle='round,pad=0.08,rounding_size=0.08',
                                      facecolor='#f8fafc', edgecolor='#1e293b', linewidth=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(x, y + 0.12, name, fontsize=11, fontweight='bold', ha='center', va='center', color='#0f172a', zorder=4)
        ax.text(x, y - 0.14, data['code'], fontsize=9.5, family='monospace', ha='center', va='center', color='#334155', zorder=4)
    for u, v, lbl in edges:
        x1, y1 = nodes[u]['pos']
        x2, y2 = nodes[v]['pos']
        dx = x2 - x1
        conn = 'arc3,rad=0.1' if abs(dx) > 0.1 else 'arc3,rad=0.0'
        arrow = patches.FancyArrowPatch((x1, y1 - 0.35), (x2, y2 + 0.35), connectionstyle=conn,
                                        arrowstyle='-|>', mutation_scale=15, color='#1d4ed8', linewidth=1.8, zorder=2)
        ax.add_patch(arrow)
        ax.text((x1 + x2)/2, (y1 + y2)/2, f' {lbl} ', fontsize=8, fontweight='bold', color='#1e40af',
                bbox=dict(boxstyle='round,pad=0.2,rounding_size=0.1', facecolor='#ffffff', edgecolor='#1d4ed8', linewidth=1), ha='center', va='center', zorder=5)
    ax.set_xlim(-1.2, 7.2)
    ax.set_ylim(-0.8, 4.8)
    ax.axis('off')
    plt.title('Dependency Graph: Tahap 1 (Base + RAW Hazard)', fontsize=12, fontweight='bold', color='#0f172a')
    plt.tight_layout()
    plt.savefig('raw_dependency_graph.png', dpi=300)
    plt.show()

# -------------------------------------------------
# Main Execution
# -------------------------------------------------
def main():
    print('*' * 60)
    print('      INSTRUCTION-LEVEL PARALLELISM SIMULATOR (BASE + RAW)')
    print('*' * 60)
    show_dependencies(instructions)
    seq_cycles = sequential_execution(instructions)
    ilp_cycles = parallel_execution(instructions)
    speedup = seq_cycles / ilp_cycles
    efficiency = (speedup / len(instructions)) * 100
    print('\n' + '=' * 50)
    print('PERFORMANCE COMPARISON')
    print('=' * 50)
    print(f'Sequential Cycles : {seq_cycles}')
    print(f'ILP Cycles        : {ilp_cycles}')
    print(f'Speedup           : {speedup:.2f}x')
    print(f'ILP Efficiency    : {efficiency:.2f}%')
    print('\nRendering Dependency Graph...')
    draw_dependency_graph()

if __name__ == '__main__':
    main()
